# 03 · Graph Neural Network Classifier
## GraphSAGE for Illicit Transaction Detection

### Why a GNN?
Classical ML models (Random Forest, XGBoost) treat nodes as independent — they ignore the graph structure entirely. A **Graph Neural Network** learns by aggregating information from a node's neighbours, its neighbours' neighbours, and so on. For fraud detection this is critical: an honest-looking transaction that receives funds from known illicit sources is itself suspicious, even if its own features look clean.

### Model: GraphSAGE
We use **GraphSAGE** (Hamilton et al., 2017) rather than a basic GCN because:
- It uses **inductive learning** — can generalise to unseen nodes (critical for the 77% unknown nodes)
- It samples a fixed-size neighbourhood at each layer, making it scalable to 200k+ nodes
- It has been applied directly to the Elliptic dataset in published research (Weber et al., 2019)

### Training strategy
- **Temporal split**: train on time steps 1–34, test on 35–49 — this mirrors real-world deployment where the model must generalise to future transactions
- **Class weighting**: compensates for the ~10:1 licit:illicit imbalance
- **Metrics**: F1 (illicit class), AUC-ROC, Precision-Recall curve — accuracy is misleading on imbalanced data

---
*References:*  
- Hamilton et al. (2017). *Inductive Representation Learning on Large Graphs*. NeurIPS.  
- Weber et al. (2019). *Anti-Money Laundering in Bitcoin*. KDD Workshop.

In [ ]:
# ── Install PyTorch Geometric (run once in Colab) ─────────────────────────────
import subprocess, sys

def run(cmd):
    subprocess.check_call(cmd, shell=True)

# Install PyG — this works on Colab with CUDA or CPU
run('pip install torch torchvision --quiet')
run('pip install torch_geometric --quiet')
run('pip install pyg_lib torch_scatter torch_sparse torch_cluster torch_spline_conv '
    '-f https://data.pyg.org/whl/torch-2.0.0+cpu.html --quiet')

print('✅ PyTorch Geometric installed')

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn   import SAGEConv
from torch_geometric.utils import from_networkx

from sklearn.metrics import (
    classification_report, roc_auc_score,
    precision_recall_curve, average_precision_score,
    confusion_matrix, RocCurveDisplay
)
from sklearn.preprocessing import StandardScaler

plt.rcParams.update({
    'figure.facecolor':'#0D1117','axes.facecolor':'#161B22',
    'axes.edgecolor':'#30363D','axes.labelcolor':'#C9D1D9',
    'xtick.color':'#8B949E','ytick.color':'#8B949E',
    'text.color':'#C9D1D9','grid.color':'#21262D',
    'grid.linestyle':'--','figure.dpi':130,
})

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
print(f'PyTorch: {torch.__version__}')

In [ ]:
# ── Load enriched features ────────────────────────────────────────────────────
full_df = pd.read_csv('../data/full_features.csv')
edges   = pd.read_csv('../data/elliptic_txs_edgelist.csv')

# Node index mapping (PyG needs 0-indexed integers)
node_list = full_df['txId'].tolist()
node_idx  = {nid: i for i, nid in enumerate(node_list)}

# Edge index tensor
valid_edges = edges[
    edges['txId1'].isin(node_idx) & edges['txId2'].isin(node_idx)
]
src = [node_idx[n] for n in valid_edges['txId1']]
dst = [node_idx[n] for n in valid_edges['txId2']]
edge_index = torch.tensor([src, dst], dtype=torch.long)

# Feature matrix
feat_cols = [c for c in full_df.columns
             if c not in ['txId','time_step','class','class_label']]

scaler = StandardScaler()
X_raw  = full_df[feat_cols].values.astype(np.float32)
X_norm = scaler.fit_transform(X_raw)
x      = torch.tensor(X_norm, dtype=torch.float)

# Labels: 1=illicit, 0=licit, -1=unknown
y_raw = full_df['class'].values
y_map = {1.0: 1, 2.0: 0, 0.0: -1}
y     = torch.tensor([y_map[v] for v in y_raw], dtype=torch.long)

# Temporal train/test masks (temporal split: steps 1-34 train, 35-49 test)
time_steps = full_df['time_step'].values
labelled   = (y >= 0)  # exclude unknown nodes from supervised training
train_mask = torch.tensor((time_steps <= 34) & labelled.numpy(), dtype=torch.bool)
test_mask  = torch.tensor((time_steps > 34)  & labelled.numpy(), dtype=torch.bool)
all_mask   = torch.ones(len(full_df), dtype=torch.bool)  # for prediction on all nodes

data = Data(x=x, edge_index=edge_index, y=y,
            train_mask=train_mask, test_mask=test_mask).to(DEVICE)

print(f'Graph loaded:')
print(f'  Nodes         : {data.num_nodes:,}')
print(f'  Edges         : {data.num_edges:,}')
print(f'  Features      : {data.num_features}')
print(f'  Train nodes   : {train_mask.sum():,}')
print(f'  Test nodes    : {test_mask.sum():,}')
print(f'  Unknown nodes : {(y==-1).sum():,}')

In [ ]:
# ── GraphSAGE model definition ─────────────────────────────────────────────────
class GraphSAGE(nn.Module):
    """
    3-layer GraphSAGE with:
      - Mean aggregation (robust to variable neighbourhood sizes)
      - BatchNorm after each layer (stabilises training on imbalanced data)
      - Dropout (prevents overfitting on the small labelled set)
      - Skip connection on layer 2→3 (helps gradient flow)
    """
    def __init__(self, in_channels, hidden_channels, out_channels, dropout=0.4):
        super().__init__()
        self.conv1 = SAGEConv(in_channels,     hidden_channels, aggr='mean')
        self.conv2 = SAGEConv(hidden_channels, hidden_channels, aggr='mean')
        self.conv3 = SAGEConv(hidden_channels, hidden_channels, aggr='mean')
        self.bn1   = nn.BatchNorm1d(hidden_channels)
        self.bn2   = nn.BatchNorm1d(hidden_channels)
        self.bn3   = nn.BatchNorm1d(hidden_channels)
        self.lin   = nn.Linear(hidden_channels, out_channels)
        self.drop  = dropout

    def forward(self, x, edge_index):
        # Layer 1
        x  = self.conv1(x, edge_index)
        x  = self.bn1(x)
        x  = F.relu(x)
        x  = F.dropout(x, p=self.drop, training=self.training)
        # Layer 2
        x2 = self.conv2(x, edge_index)
        x2 = self.bn2(x2)
        x2 = F.relu(x2)
        x2 = F.dropout(x2, p=self.drop, training=self.training)
        # Layer 3 with residual
        x3 = self.conv3(x2, edge_index)
        x3 = self.bn3(x3 + x2)  # skip connection
        x3 = F.relu(x3)
        x3 = F.dropout(x3, p=self.drop, training=self.training)
        return self.lin(x3)

    def embed(self, x, edge_index):
        """Return node embeddings (pre-classification layer) for visualisation."""
        with torch.no_grad():
            self.eval()
            x  = F.relu(self.bn1(self.conv1(x, edge_index)))
            x2 = F.relu(self.bn2(self.conv2(x, edge_index)))
            x3 = F.relu(self.bn3(self.conv3(x2, edge_index) + x2))
        return x3


# Class weights to handle imbalance
n_licit   = int((data.y[data.train_mask] == 0).sum())
n_illicit = int((data.y[data.train_mask] == 1).sum())
w_illicit = n_licit / n_illicit  # ~10x weight on illicit class
class_weights = torch.tensor([1.0, w_illicit], dtype=torch.float).to(DEVICE)

model = GraphSAGE(
    in_channels     = data.num_features,
    hidden_channels = 128,
    out_channels    = 2,
    dropout         = 0.4
).to(DEVICE)

optimiser = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=5e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimiser, patience=10, factor=0.5)
criterion = nn.CrossEntropyLoss(weight=class_weights)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model parameters: {total_params:,}')
print(f'Class weight (illicit): {w_illicit:.2f}x')
print(model)

In [ ]:
# ── Training loop ──────────────────────────────────────────────────────────────
def train_epoch():
    model.train()
    optimiser.zero_grad()
    out  = model(data.x, data.edge_index)
    loss = criterion(out[data.train_mask], data.y[data.train_mask])
    loss.backward()
    optimiser.step()
    return loss.item()

@torch.no_grad()
def evaluate(mask):
    model.eval()
    out   = model(data.x, data.edge_index)
    probs = F.softmax(out, dim=1)[:, 1]  # probability of illicit
    preds = out.argmax(dim=1)
    y_true = data.y[mask].cpu().numpy()
    y_pred = preds[mask].cpu().numpy()
    y_prob = probs[mask].cpu().numpy()
    auc = roc_auc_score(y_true, y_prob) if len(np.unique(y_true)) > 1 else 0.0
    f1  = classification_report(y_true, y_pred, output_dict=True, zero_division=0)['1']['f1-score']
    return f1, auc

EPOCHS = 200
history = {'train_loss':[], 'val_f1':[], 'val_auc':[]}
best_f1, best_state = 0, None

for epoch in range(1, EPOCHS+1):
    loss = train_epoch()
    f1, auc = evaluate(data.test_mask)
    scheduler.step(1 - f1)   # maximise F1
    history['train_loss'].append(loss)
    history['val_f1'].append(f1)
    history['val_auc'].append(auc)

    if f1 > best_f1:
        best_f1    = f1
        best_state = {k: v.clone() for k, v in model.state_dict().items()}

    if epoch % 20 == 0:
        print(f'Epoch {epoch:3d} | Loss: {loss:.4f} | F1: {f1:.4f} | AUC: {auc:.4f}')

# Restore best model
model.load_state_dict(best_state)
torch.save(best_state, '../data/graphsage_best.pt')
print(f'\n✅ Training complete. Best F1 (illicit): {best_f1:.4f}')

In [ ]:
# ── Training curves ────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('GraphSAGE Training Dynamics', fontsize=14)

axes[0].plot(history['train_loss'], color='#4A90D9', lw=1.5)
axes[0].set_title('Training Loss (Cross-Entropy)')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')

axes[1].plot(history['val_f1'], color='#FF4444', lw=1.5)
axes[1].axhline(best_f1, color='#FFD700', ls='--', lw=1, label=f'Best: {best_f1:.4f}')
axes[1].set_title('Test F1 Score (Illicit Class)')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('F1')
axes[1].legend()

axes[2].plot(history['val_auc'], color='#00C9A7', lw=1.5)
axes[2].set_title('Test AUC-ROC')
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('AUC')

plt.tight_layout()
plt.savefig('../data/fig_training_curves.png', bbox_inches='tight', dpi=130)
plt.show()

In [ ]:
# ── Full evaluation ────────────────────────────────────────────────────────────
model.eval()
with torch.no_grad():
    out   = model(data.x, data.edge_index)
    probs = F.softmax(out, dim=1)[:, 1].cpu().numpy()
    preds = out.argmax(dim=1).cpu().numpy()

y_test      = data.y[data.test_mask].cpu().numpy()
preds_test  = preds[data.test_mask.cpu().numpy()]
probs_test  = probs[data.test_mask.cpu().numpy()]

print('=== Classification Report (Test Set: Time Steps 35–49) ===')
print(classification_report(y_test, preds_test,
                             target_names=['Licit','Illicit'], zero_division=0))

auc = roc_auc_score(y_test, probs_test)
ap  = average_precision_score(y_test, probs_test)
print(f'AUC-ROC           : {auc:.4f}')
print(f'Avg Precision (AP): {ap:.4f}')

# Confusion matrix
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('Model Evaluation — Test Set', fontsize=13)

cm = confusion_matrix(y_test, preds_test)
sns_cm = sns.heatmap(cm, annot=True, fmt='d', ax=axes[0],
                     xticklabels=['Licit','Illicit'],
                     yticklabels=['Licit','Illicit'],
                     cmap='Blues', linewidths=1)
axes[0].set_title('Confusion Matrix')
axes[0].set_ylabel('True Label')
axes[0].set_xlabel('Predicted Label')

# Precision-Recall curve (more informative than ROC for imbalanced data)
precision, recall, thresholds = precision_recall_curve(y_test, probs_test)
axes[1].plot(recall, precision, color='#FF4444', lw=2,
             label=f'AP = {ap:.4f}')
axes[1].axhline(n_illicit/(n_illicit+n_licit), color='#8B949E',
                ls='--', lw=1, label='Random baseline')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_title('Precision-Recall Curve (Illicit Class)')
axes[1].legend()

plt.tight_layout()
plt.savefig('../data/fig_evaluation.png', bbox_inches='tight', dpi=130)
plt.show()

In [ ]:
# ── Predict on unknown nodes ───────────────────────────────────────────────────
# THIS is the key research contribution: labelling the 77% unknown wallets

unknown_mask = (full_df['class'] == 0).values
unknown_probs = probs[unknown_mask]

unknown_df = full_df[unknown_mask][['txId','time_step']].copy()
unknown_df['illicit_probability'] = unknown_probs
unknown_df['predicted_label']     = np.where(unknown_probs >= 0.5, 'Illicit', 'Licit')
unknown_df = unknown_df.sort_values('illicit_probability', ascending=False)

n_pred_illicit = (unknown_df['predicted_label']=='Illicit').sum()
print(f'Predictions on {len(unknown_df):,} unknown wallets:')
print(f'  Predicted Illicit : {n_pred_illicit:,} ({n_pred_illicit/len(unknown_df)*100:.1f}%)')
print(f'  Predicted Licit   : {len(unknown_df)-n_pred_illicit:,}')
print(f'\nTop 20 highest-risk previously-unknown wallets:')
print(unknown_df.head(20).to_string(index=False))

# Save predictions
unknown_df.to_csv('../data/unknown_node_predictions.csv', index=False)
print('\n✅ Saved → ../data/unknown_node_predictions.csv')

In [ ]:
# ── Node embeddings visualisation (t-SNE) ─────────────────────────────────────
# Shows whether the GNN has learned a meaningful latent space
from sklearn.manifold import TSNE

print('Computing node embeddings…')
with torch.no_grad():
    embeddings = model.embed(data.x, data.edge_index).cpu().numpy()

# Sample for t-SNE (full graph is too large)
SAMPLE = 3000
labelled_idx = np.where(full_df['class'].isin([1.0,2.0]))[0]
sample_idx   = np.random.default_rng(42).choice(labelled_idx,
                                                  size=min(SAMPLE, len(labelled_idx)),
                                                  replace=False)
emb_sample   = embeddings[sample_idx]
lbl_sample   = full_df.iloc[sample_idx]['class'].map({1.0:'Illicit',2.0:'Licit'}).values

print('Running t-SNE…')
tsne   = TSNE(n_components=2, perplexity=30, random_state=42, n_iter=500)
coords = tsne.fit_transform(emb_sample)

fig, ax = plt.subplots(figsize=(10, 7))
for lbl, colour in [('Licit','#00C9A7'),('Illicit','#FF4444')]:
    mask = lbl_sample == lbl
    ax.scatter(coords[mask,0], coords[mask,1],
               c=colour, s=8, alpha=0.6, label=lbl, edgecolors='none')
ax.set_title('t-SNE of GraphSAGE Node Embeddings\n'
             '(Good separation = model learned meaningful fraud structure)',
             fontsize=12)
ax.legend(markerscale=3)
ax.set_xticks([])
ax.set_yticks([])

plt.tight_layout()
plt.savefig('../data/fig_tsne_embeddings.png', bbox_inches='tight', dpi=130)
plt.show()
print('\nGNN classifier complete. Proceed to 04_Temporal_Analysis.ipynb')